## old code (from Claude)

In [ ]:
import re
from collections import Counter

import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer, ENGLISH_STOP_WORDS

# ---- 1. Settings, then load your cleaned data -------------------------------
DATA_PATH = "goodreads_onestar_clean.csv"   # your cleaned file
TEXT_COL = "review_text"   # must be the cleaned-but-NOT-normalized text:
                           # keep "not", stopwords, and original wording
MIN_WORDS = 15             # skip reviews shorter than this (optional)
SEED = 42

# Book titles, authors, and character names to hide from the word comparison.
# Add more after your first run if titles/authors clutter the results.
MASK = ["alice", "wonderland", "carroll", "silverstein", "peter"]

# If your cleaned data is already in a DataFrame in your notebook, replace the
# next line with:  one_star = df.copy()
one_star = pd.read_csv(DATA_PATH)
one_star = one_star.dropna(subset=[TEXT_COL])
one_star = one_star[one_star[TEXT_COL].str.split().str.len() >= MIN_WORDS]

print(f"1-star reviews: {len(one_star):,}")


# ---- 2. The comparison method (Fightin' Words) ------------------------------
def compare_groups(texts_a, texts_b):
    """Score every word and phrase.
    Positive score = more typical of group A. Negative = more typical of group B.
    A score beyond about +2 or -2 is probably a real difference."""
    texts_a = list(texts_a)
    texts_b = list(texts_b)

    # Count single words and two-word phrases, ignoring very rare ones
    counter = CountVectorizer(
        ngram_range=(1, 2),
        min_df=20,                              # must appear in 20+ reviews
        token_pattern=r"(?u)\b\w[\w']*\b",      # keeps "don't" as one word
    )
    counts = counter.fit_transform(texts_a + texts_b)

    # Total count of each term in each group
    n_a = len(texts_a)
    count_a = np.asarray(counts[:n_a].sum(axis=0)).ravel()
    count_b = np.asarray(counts[n_a:].sum(axis=0)).ravel()

    # Baseline: how common each term is overall (this is the "prior")
    baseline = count_a + count_b
    baseline_total = baseline.sum()

    # How strongly each term leans toward each group (log-odds)
    odds_a = np.log((count_a + baseline) / (count_a.sum() + baseline_total - count_a - baseline))
    odds_b = np.log((count_b + baseline) / (count_b.sum() + baseline_total - count_b - baseline))
    difference = odds_a - odds_b

    # Turn the difference into a score that accounts for how much data there is
    uncertainty = np.sqrt(1 / (count_a + baseline) + 1 / (count_b + baseline))
    score = difference / uncertainty

    results = pd.DataFrame({
        "term": counter.get_feature_names_out(),
        "score": score,
        "count_A": count_a,
        "count_B": count_b,
    })

    # Hide titles, authors, and character names
    for name in MASK:
        results = results[~results["term"].str.contains(r"\b" + name + r"\b")]

    return results.sort_values("score", ascending=False).reset_index(drop=True)


# ---- 3. What counts as an "inappropriateness" complaint ---------------------
# Wording that signals an objection. Phrases like "not age appropriate" are
# included, but plain "age appropriate" is not, because that is often praise.
TRIGGER = (
    r"(?:age[- ])?inappropriate\w*"
    r"|\b(?:not|isn't|wasn't|aren't|weren't)\s+(?:\w+\s+){0,2}(?:age[- ])?appropriate"
    r"|unsuitable|not suitable|offensive|offend\w*"
)
WINDOW = 10   # how many words on each side of the trigger to look at

# ---- 4. Find the matching reviews -------------------------------------------
has_trigger = one_star[TEXT_COL].str.contains(TRIGGER, case=False, regex=True)
matched = one_star[has_trigger]
others = one_star[~has_trigger]
print(f"{len(matched):,} of {len(one_star):,} 1-star reviews contain an "
      f"inappropriateness complaint ({len(matched) / len(one_star):.1%})")


# ---- 5. Read the context: the words around each complaint -------------------
def snippet(text, width=12):
    """The words just before and after the first trigger, with it in [[ ]]."""
    m = re.search(TRIGGER, text, flags=re.I)
    before = " ".join(text[:m.start()].split()[-width:])
    after = " ".join(text[m.end():].split()[:width])
    return f"{before} [[{m.group(0)}]] {after}"


matched = matched.copy()
matched["snippet"] = matched[TEXT_COL].apply(snippet)

pd.set_option("display.max_colwidth", None)
print("\n25 random examples (read these first):")
print(matched[["book_id", "snippet"]].sample(min(25, len(matched)),
                                             random_state=SEED).to_string())

# ---- 6. Which words show up near the complaints? ----------------------------
skip = set(ENGLISH_STOP_WORDS) | {"book", "books", "read", "story", "kids", "child",
                                  "children", "just", "like", "really", "don",
                                  "didn", "appropriate", "inappropriate",
                                  "offensive", "suitable", "unsuitable", "offend"}
nearby = Counter()
for text in matched[TEXT_COL].str.lower():
    for m in re.finditer(TRIGGER, text):
        before = re.findall(r"[a-z']+", text[:m.start()])[-WINDOW:]
        after = re.findall(r"[a-z']+", text[m.end():])[:WINDOW]
        nearby.update(w for w in before + after if w not in skip and len(w) > 2)

print(f"\nWords most often within {WINDOW} words of a complaint:")
print(pd.DataFrame(nearby.most_common(40), columns=["word", "times"]).to_string())

# ---- 7. Which words are distinctive to these reviews? -----------------------
# The list above includes words that are common everywhere. This compares the
# matching 1-star reviews with the other 1-star reviews, so only words that
# stand out are left.
distinct = compare_groups(matched[TEXT_COL], others[TEXT_COL])
distinct = distinct[~distinct["term"].str.contains("appropriat|suitab|offen")]
distinct.to_csv("words_inappropriate_vs_other_1star.csv", index=False)
print("\nMost distinctive words in reviews that call something inappropriate:")
print(distinct.head(40).round(2).to_string())

# ---- 8. Save a sample to hand-code ------------------------------------------
# Open the CSV, and in the "reason" column write what each reviewer objects to
# (e.g. "violence", "language", "sexual content", "message", "illustrations").
to_code = matched[["book_id", TEXT_COL, "snippet"]].sample(
    min(200, len(matched)), random_state=SEED)
to_code["reason"] = ""
to_code.to_csv("inappropriate_to_code.csv", index=False)
print("\nSaved inappropriate_to_code.csv with", len(to_code), "reviews to label")


# ---- 9. What are the NOT-flagged reviews about? -----------------------------
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import NMF

# 9a. Words most typical of the other reviews (the bottom of the Section 7 table)
print("\nWords most typical of the NOT-flagged reviews:")
print(distinct.sort_values("score").head(40).round(2).to_string(index=False))

# 9b. Random examples to read
print("\n15 random NOT-flagged reviews (first 300 characters):")
for text in others[TEXT_COL].sample(15, random_state=SEED):
    print("-", text[:300].replace("\n", " "), "\n")

# 9c. Topic model on the NOT-flagged reviews
N_TOPICS = 8
tfidf = TfidfVectorizer(stop_words="english", min_df=10, max_df=0.5)
X = tfidf.fit_transform(others[TEXT_COL])
nmf = NMF(n_components=N_TOPICS, random_state=SEED, init="nndsvda", max_iter=400)
doc_topics = nmf.fit_transform(X)
terms = tfidf.get_feature_names_out()

others = others.copy()
others["topic"] = doc_topics.argmax(axis=1)
for i, weights in enumerate(nmf.components_):
    top_words = [terms[j] for j in weights.argsort()[::-1][:10]]
    n_reviews = (others["topic"] == i).sum()
    print(f"\nTopic {i} ({n_reviews:,} reviews): {', '.join(top_words)}")